# Demo of KG with LLM applications

## KG construction with Neo4j 

In [1]:
import os, sys
import re
# Settings #
cwd = os.getcwd()
frameworkDir = os.path.abspath(os.path.join(cwd, os.pardir, os.pardir, 'src'))
sys.path.append(frameworkDir)
print(f"Add path: {frameworkDir} to system path")
#######################
# Internal Modules #
from dackar.knowledge_graph.py2neo import Py2Neo
from dackar.knowledge_graph.graph_utils import set_neo4j_import_folder

# uri = "bolt://localhost:7687" # for a single instance
uri = "neo4j://localhost:7687" # for a cluster
pwd = "123456789" # user need to provide the DBMS database password

py2neo = Py2Neo(uri=uri, user='neo4j', pwd=pwd)
py2neo.reset()
# py2neo.close()

Add path: /Users/wangc/projects/DACKAR/src to system path


/Users/wangc/miniforge3/envs/dackar_libs/lib/python3.11/site-packages/quantulum3/classifier.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
### Load MBSE model node data
file_path = 'test_nodes.csv'
label = 'MBSE'
attribute = {'nodeId':'nodeId', 'label':'label', 'ID':'ID', 'type':'type'}
py2neo.load_csv_for_nodes(file_path, label, attribute)
### Load MBSE model relationship data
file_path = 'test_edges.csv'
l1='MBSE'
p1={'nodeId':'sourceNodeId'}
l2='MBSE'
p2 ={'nodeId':'targetNodeId'}
lr = 'MBSE_link'
pr = {'prop':'type'}
py2neo.load_csv_for_relations(file_path, l1, p1, l2, p2, lr, pr)
### Load monitoring data
file_path = 'test_monit_vars.csv'
label = 'monitor_var'
attribute = {'nodeId':'varID'}
py2neo.load_csv_for_nodes(file_path, label, attribute)
### Load monitoring relationship data and link to MBSE model data
file_path = 'test_monit_vars.csv'
l1='monitor_var'
p1={'nodeId':'varID'}
l2='MBSE'
p2 ={'nodeId':'MBSE_ID'}
lr = 'monitoring'
pr = None
py2neo.load_csv_for_relations(file_path, l1, p1, l2, p2, lr, pr)
### Load anomaly detection data
file_path = 'test_AD.csv'
label = 'anomaly_detect'
attribute = {'nodeId':'AD_ID', 'type':'type'}
py2neo.load_csv_for_nodes(file_path, label, attribute)
### Load anomaly detection relation data and link to monitoring data
file_path = 'test_AD.csv'
l1='anomaly_detect'
p1={'nodeId':'AD_ID'}
l2='monitor_var'
p2 ={'nodeId':'var_ID'}
lr = 'input_from'
pr = None
py2neo.load_csv_for_relations(file_path, l1, p1, l2, p2, lr, pr)
### Load anomalies
file_path = 'test_anomalies.csv'
label = 'anomaly'
attribute = {'nodeId':'anom_ID', 'time_initial':'t_in', 'time_final':'t_fin'}
py2neo.load_csv_for_nodes(file_path, label, attribute)
### Load anomalies relation data and link to anomaly detection method
file_path = 'test_anomalies.csv'
l1='anomaly'
p1={'nodeId':'anom_ID'}
l2='anomaly_detect'
p2 ={'nodeId':'AD_ID'}
lr = 'detected_by'
pr = None
py2neo.load_csv_for_relations(file_path, l1, p1, l2, p2, lr, pr)

## LLM Integration

In [3]:
# Install Dackar libraries first
# Then install LLM related libraries
# !pip install openai langchain langchain_openai langchain_community langchain_neo4j
# !pip install langchain-core ollama langchain-ollama

In [ ]:
# Set Neo4j Graph and LLM chat and embedding model

from langchain_neo4j import Neo4jGraph
from langchain_ollama.chat_models import ChatOllama
from langchain_ollama.embeddings import OllamaEmbeddings

chat_model = ChatOllama(
    model="gpt-oss:20b",          # must be pulled in Ollama
    temperature=0,             # pass any Ollama params here
    base_url="http://localhost:11434",  # default; override if needed
)

embedding_model = OllamaEmbeddings(model="nomic-embed-text")

# LangChain's interface to Neo4j Database
graph = Neo4jGraph(
    url=uri, username="neo4j", password=pwd
)

graph.structured_schema

/Users/wangc/miniforge3/envs/dackar_libs/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'node_props': {'MBSE': [{'property': 'ID', 'type': 'STRING'},
   {'property': 'label', 'type': 'STRING'},
   {'property': 'nodeId', 'type': 'STRING'},
   {'property': 'type', 'type': 'STRING'}],
  'monitor_var': [{'property': 'nodeId', 'type': 'STRING'}],
  'anomaly_detect': [{'property': 'nodeId', 'type': 'STRING'},
   {'property': 'type', 'type': 'STRING'}],
  'anomaly': [{'property': 'nodeId', 'type': 'STRING'},
   {'property': 'time_final', 'type': 'STRING'},
   {'property': 'time_initial', 'type': 'STRING'}]},
 'rel_props': {'MBSE_link': [{'property': 'prop', 'type': 'STRING'}]},
 'relationships': [{'start': 'MBSE', 'type': 'MBSE_link', 'end': 'MBSE'},
  {'start': 'monitor_var', 'type': 'monitoring', 'end': 'MBSE'},
  {'start': 'anomaly_detect', 'type': 'input_from', 'end': 'monitor_var'},
  {'start': 'anomaly', 'type': 'detected_by', 'end': 'anomaly_detect'}],
 'metadata': {'constraint': [{'id': 9,
    'name': 'affectedorganism',
    'type': 'UNIQUENESS',
    'entityType': 'NODE

#### Basic Graph Query with LLM

**Query Process:**
1. User asks natural language question
2. LLM generates Cypher query
3. Query executes against Neo4j
4. Results are returned and optionally formatted

In [8]:
from langchain_neo4j.chains.graph_qa.cypher import GraphCypherQAChain
chain_1 = GraphCypherQAChain.from_llm(
    chat_model,
    graph=graph,
    verbose=True,
    return_intermediate_steps=True,
    return_direct=True,
    allow_dangerous_requests=True,
)

In [9]:
chain_1.invoke({"query": "return the MBSE nodes?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:MBSE) RETURN m;

> Finished chain.


{'query': 'return the MBSE nodes?',
 'result': [{'m': {'ID': 'C3',
    'label': 'C3',
    'type': 'entity_emb',
    'nodeId': '0'}},
  {'m': {'ID': 'cond1',
    'label': 'condenser',
    'type': 'entity',
    'nodeId': '1'}},
  {'m': {'ID': 'I_1PC9PKV4BTGV5_BMXNGD2MYES4D',
    'label': 'pipe',
    'type': 'LML_link',
    'nodeId': '2'}},
  {'m': {'ID': 'I_9YG45MX0QMK91_80WEVYKF2ZK18',
    'label': 'pipe',
    'type': 'LML_link',
    'nodeId': '3'}},
  {'m': {'ID': 'V2', 'label': 'V2', 'type': 'entity_emb', 'nodeId': '4'}},
  {'m': {'ID': 'V3', 'label': 'V3', 'type': 'entity_emb', 'nodeId': '5'}},
  {'m': {'ID': 'S3',
    'label': 'level sensor',
    'type': 'entity_emb',
    'nodeId': '6'}},
  {'m': {'ID': 'None', 'label': 'forebay', 'type': 'entity', 'nodeId': '7'}},
  {'m': {'ID': 'body',
    'label': 'OPM_pump',
    'type': 'MBSE_linked_ent',
    'nodeId': '8'}},
  {'m': {'ID': 'PM1', 'label': 'pump', 'type': 'entity', 'nodeId': '9'}}],
 'intermediate_steps': [{'query': 'MATCH (m:MB

In [10]:
chain_1.invoke({"query": "return all relations for node with label pump?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (n:MBSE {label: 'pump'})-[r]-()
RETURN n, r;

> Finished chain.


{'query': 'return all relations for node with label pump?',
 'result': [{'n': {'ID': 'PM1',
    'label': 'pump',
    'type': 'entity',
    'nodeId': '9'},
   'r': ({'ID': 'PM1', 'label': 'pump', 'type': 'entity', 'nodeId': '9'},
    'MBSE_link',
    {})},
  {'n': {'ID': 'PM1', 'label': 'pump', 'type': 'entity', 'nodeId': '9'},
   'r': ({'ID': 'PM1', 'label': 'pump', 'type': 'entity', 'nodeId': '9'},
    'MBSE_link',
    {})},
  {'n': {'ID': 'PM1', 'label': 'pump', 'type': 'entity', 'nodeId': '9'},
   'r': ({},
    'MBSE_link',
    {'ID': 'PM1', 'label': 'pump', 'type': 'entity', 'nodeId': '9'})},
  {'n': {'ID': 'PM1', 'label': 'pump', 'type': 'entity', 'nodeId': '9'},
   'r': ({},
    'monitoring',
    {'ID': 'PM1', 'label': 'pump', 'type': 'entity', 'nodeId': '9'})},
  {'n': {'ID': 'PM2', 'label': 'pump', 'type': 'entity', 'nodeId': '12'},
   'r': ({'ID': 'PM2', 'label': 'pump', 'type': 'entity', 'nodeId': '12'},
    'MBSE_link',
    {})},
  {'n': {'ID': 'PM2', 'label': 'pump', 'type'

In [11]:
chain_1.invoke({"query": "what are the anomalies associate to variable x_2?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (mv:monitor_var {nodeId: 'x_2'})<-[:input_from]-(ad:anomaly_detect)<-[:detected_by]-(a:anomaly)
RETURN a.nodeId AS anomalyId, a.time_initial, a.time_final;

> Finished chain.


{'query': 'what are the anomalies associate to variable x_2?',
 'result': [{'anomalyId': '2', 'a.time_initial': '5.5', 'a.time_final': '6'},
  {'anomalyId': '4', 'a.time_initial': '10', 'a.time_final': '10.5'}],
 'intermediate_steps': [{'query': "MATCH (mv:monitor_var {nodeId: 'x_2'})<-[:input_from]-(ad:anomaly_detect)<-[:detected_by]-(a:anomaly)\nRETURN a.nodeId AS anomalyId, a.time_initial, a.time_final;"}]}

In [12]:
chain_1.invoke({"query": "what are the nodes connected to pump PM1?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (pump:MBSE {ID: "PM1"})-[:MBSE_link]->(connected)
RETURN DISTINCT connected
UNION
MATCH (connected)-[:MBSE_link]->(pump)
RETURN DISTINCT connected
UNION
MATCH (connected)-[:monitoring]->(pump)
RETURN DISTINCT connected

> Finished chain.


{'query': 'what are the nodes connected to pump PM1?',
 'result': [{'connected': {'ID': 'body',
    'label': 'OPM_pump',
    'type': 'MBSE_linked_ent',
    'nodeId': '8'}},
  {'connected': {'ID': 'I_2J6B8DXJ2WH0N_AR6M5HTA4X3SZ',
    'label': 'pipe',
    'type': 'LML_link',
    'nodeId': '10'}},
  {'connected': {'ID': 'cond1',
    'label': 'condenser',
    'type': 'entity',
    'nodeId': '1'}},
  {'connected': {'ID': 'I_1PC9PKV4BTGV5_BMXNGD2MYES4D',
    'label': 'pipe',
    'type': 'LML_link',
    'nodeId': '2'}},
  {'connected': {'ID': 'I_9YG45MX0QMK91_80WEVYKF2ZK18',
    'label': 'pipe',
    'type': 'LML_link',
    'nodeId': '3'}},
  {'connected': {'ID': 'None',
    'label': 'forebay',
    'type': 'entity',
    'nodeId': '7'}},
  {'connected': {'ID': 'PM1',
    'label': 'pump',
    'type': 'entity',
    'nodeId': '9'}},
  {'connected': {'ID': 'PM2',
    'label': 'pump',
    'type': 'entity',
    'nodeId': '12'}},
  {'connected': {'ID': 'I_2G6275A7X6HT5_86DMC9T834MZG',
    'label': 'pi

## Enhanced Query Chain with Custom Prompts

**Enhanced Features:**
- **Query Decomposition**: Breaks down complex questions into steps
- **Domain Knowledge**: Includes relationship patterns
- **Case Insensitivity**: Handles variations in entity names
- **Fuzzy Matching**: Uses CONTAINS for partial name matches
- **ML Predictions**: Includes machine learning predicted relationships
- **Validation**: Checks Cypher syntax before execution

In [13]:
prompt_graph_schema = f"Nodes: {graph.structured_schema['node_props'].keys()} \nRelationships: {graph.structured_schema['relationships']}".replace(
    "{", "{{"
).replace(
    "}", "}}"
)

In [14]:
prompt_graph_schema

"Nodes: dict_keys(['MBSE', 'monitor_var', 'anomaly_detect', 'anomaly']) \nRelationships: [{{'start': 'MBSE', 'type': 'MBSE_link', 'end': 'MBSE'}}, {{'start': 'monitor_var', 'type': 'monitoring', 'end': 'MBSE'}}, {{'start': 'anomaly_detect', 'type': 'input_from', 'end': 'monitor_var'}}, {{'start': 'anomaly', 'type': 'detected_by', 'end': 'anomaly_detect'}}]"

### Construct QA prompt template and Cypher generation template 

- Using GraphCypherQAChain
- Zero-short learning

In [17]:
from langchain_core.prompts.prompt import PromptTemplate

QA_TEMPLATE = """
Before generating the cypher query, always decompose what the final query should do.
Question: {question}
"""

QA_PROMPT = PromptTemplate(input_variables=["question"], template=QA_TEMPLATE)


CYPHER_GENERATION_TEMPLATE = (
    """Task:Generate Cypher statement to query a graph database.
Instructions:
Use only the provided relationship types and properties in the schema.
Do not use any other relationship types or properties that are not provided.
Schema:
"""
    + prompt_graph_schema
    + """
Note: Do not include any explanations or apologies in your responses.
Do not respond to any questions that might ask anything else than for you to construct a Cypher statement.
Do not include any text except the generated Cypher statement.

For complex queries think of the relationships required for execution from the list above.
#What are the possible causes for the failure of MBSE pump PM1. To answer this question several steps should be provided:
1. monitor_var - monitoring -> MBSE pump PM1
2. anomaly_detect - input_from -> monitor_var
3. anomaly - detected_by -> anomaly_detect
4. anomaly is the possible cause of the MBSE pump PM1 failure
MATCH (m:MBSE {{label: 'pump', ID: 'PM1'}})<-[:monitoring]-(mv:monitor_var)<-[:input_from]-(ad:anomaly_detect)<-[:detected_by]-(a:anomaly) return a

Always use toLower method to avoid case sensitivity:
# Which variables monitoring pump PM1?
MATCH (m:MBSE)<-[:monitoring]-(mv:monitor_var) WHERE toLower(m.label) CONTAINS('pump') AND toLower(m.ID) CONTAINS('pm1') return mv
# List pump PM1 monitoring variables
MATCH (m:MBSE)<-[:monitoring]-(mv:monitor_var) WHERE toLower(m.label) CONTAINS('pump') AND toLower(m.ID) CONTAINS('pm1') return mv
# List PM1 monitoring variables
MATCH (m:MBSE)<-[:monitoring]-(mv:monitor_var) WHERE toLower(m.label) CONTAINS('pm1') OR toLower(m.ID) CONTAINS('pm1') return mv

The question is:
{question}"""
)

CYPHER_GENERATION_PROMPT = PromptTemplate(
    input_variables=["question"], template=CYPHER_GENERATION_TEMPLATE
)

chain_2 = GraphCypherQAChain.from_llm(
    chat_model,
    graph=graph,
    verbose=True,
    return_intermediate_steps=True,
    return_direct=True,
    validate_cypher=True,
    use_function_response=True,
    qa_prompt=QA_PROMPT,
    cypher_prompt=CYPHER_GENERATION_PROMPT,
    allow_dangerous_requests=True,
)

In [18]:
chain_2.invoke({"query":"What are the possible causes for the failure of MBSE pump PM1"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:MBSE)<-[:monitoring]-(mv:monitor_var)<-[:input_from]-(ad:anomaly_detect)<-[:detected_by]-(a:anomaly) WHERE toLower(m.label) CONTAINS('pump') AND toLower(m.ID) CONTAINS('pm1') RETURN a

> Finished chain.


{'query': 'What are the possible causes for the failure of MBSE pump PM1',
 'result': [{'a': {'time_initial': '1', 'nodeId': '1', 'time_final': '2'}},
  {'a': {'time_initial': '10', 'nodeId': '4', 'time_final': '10.5'}}],
 'intermediate_steps': [{'query': "MATCH (m:MBSE)<-[:monitoring]-(mv:monitor_var)<-[:input_from]-(ad:anomaly_detect)<-[:detected_by]-(a:anomaly) WHERE toLower(m.label) CONTAINS('pump') AND toLower(m.ID) CONTAINS('pm1') RETURN a"}]}

In [19]:
chain_2.invoke({"query":"Which variables monitoring pump PM2?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:MBSE)<-[:monitoring]-(mv:monitor_var) 
WHERE toLower(m.label) CONTAINS('pump') AND toLower(m.ID) CONTAINS('pm2') 
RETURN mv;

> Finished chain.


{'query': 'Which variables monitoring pump PM2?',
 'result': [{'mv': {'nodeId': 'x_2'}}],
 'intermediate_steps': [{'query': "MATCH (m:MBSE)<-[:monitoring]-(mv:monitor_var) \nWHERE toLower(m.label) CONTAINS('pump') AND toLower(m.ID) CONTAINS('pm2') \nRETURN mv;"}]}

## Semantic Search or Vector Search Setup

**Vector Search Benefits:**
- **Semantic Understanding**: Finds related concepts, not just exact matches
- **Fuzzy Matching**: Handles typos and variations in medical terms
- **Ranked Results**: Returns similarity scores for relevance
- **Domain Flexibility**: Works with synonyms and related medical conditions